In [1]:
!rm -rf /kaggle/working/* 

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
import torch
import os 
import warnings
warnings.filterwarnings("ignore")
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import torch
import gc
import os
import shutil
# HuggingFace
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from datasets import Dataset
os.environ["TOKENIZERS_PARALLELISM"] = "false"   # Kills "fork" warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"        # Kills TensorFlow logs (if any)
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"  

# Check GPU
device = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"Using device: {device}")
%matplotlib inline

2025-12-15 14:32:42.551029: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765809162.572060    7050 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765809162.578472    7050 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

Using device: Tesla P100-PCIE-16GB


In [3]:
train=pd.read_csv('/kaggle/input/social-media-extremism-detection-challenge/train.csv')
test=pd.read_csv('/kaggle/input/social-media-extremism-detection-challenge/test.csv')
print(f"Train Data Shape: {train.shape}")
print(f"Test Data Shape: {test.shape}")
print(f"Check Null Values in Train Data: {train.isnull().sum()}")
print(f"Check Null Values in Test Data: {test.isnull().sum()}")

Train Data Shape: (2250, 3)
Test Data Shape: (750, 2)
Check Null Values in Train Data: ID                  0
Original_Message    1
Extremism_Label     0
dtype: int64
Check Null Values in Test Data: ID                  0
Original_Message    0
dtype: int64


In [4]:
train = train.rename(columns={"Original_Message": "text","Extremism_Label":"label"})
train['text'] = train['text'].fillna(train['text'].mode()[0])
train.head()

,ID,text,label
0,1,sixth forms should burn to the ground,EXTREMIST
1,2,whatever should burn benders to the ground,EXTREMIST
2,3,factories should burn to the ground,EXTREMIST
3,4,halal factories should burn to the ground,EXTREMIST
4,5,nhs should burn to the ground,EXTREMIST


In [5]:
label2id = {"NON_EXTREMIST": 0, "EXTREMIST": 1}
id2label = {0: "NON_EXTREMIST", 1: "EXTREMIST"}

train['label'] = train['label'].map(label2id)

# Very light cleaning – only for stability
def clean_text(text):
    text = text.lower()
    text = re.sub(r'@\w+', 'user', text)
    text = re.sub(r'http\S+', 'url', text)
    return text.strip()

train['text'] = train['text'].apply(clean_text)
test['text'] = test['Original_Message'].apply(clean_text)

In [6]:
model_name = "Hate-speech-CNERG/dehatebert-mono-english"  

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

# Convert to HuggingFace Dataset
train_hf = Dataset.from_pandas(train[['text', 'label']])
test_hf = Dataset.from_pandas(test[['text']])

tokenized_train = train_hf.map(tokenize_function, batched=True)
tokenized_test = test_hf.map(tokenize_function, batched=True)

# Remove unnecessary columns and set format
tokenized_train = tokenized_train.remove_columns(["text"])
tokenized_test = tokenized_test.remove_columns(["text"])
tokenized_train.set_format("torch")
tokenized_test.set_format("torch")

tokenizer_config.json:   0%|          | 0.00/152 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/2250 [00:00<?, ? examples/s]

Map:   0%|          | 0/750 [00:00<?, ? examples/s]

In [7]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros(len(train))
test_preds = np.zeros(len(test))

for fold, (train_idx, val_idx) in enumerate(skf.split(train, train['label'])):
    print(f"\n=== Fold {fold+1} ===")

    train_fold = tokenized_train.select(train_idx.tolist())
    val_fold = tokenized_train.select(val_idx.tolist())

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2, id2label=id2label, label2id=label2id)

    args = TrainingArguments(
        output_dir=f"./dehatebert_fold{fold}",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=3e-5,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=64,
        num_train_epochs=10,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        greater_is_better=True,
        save_total_limit=1,
        save_safetensors=True,
        fp16=True,
        seed=42,
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        logging_steps=20,
        report_to="none",
        remove_unused_columns=False,
        dataloader_num_workers=2,
        disable_tqdm=False,
    )

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=1)
        return {"accuracy": accuracy_score(labels, preds)}

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_fold,
        eval_dataset=val_fold,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    val_pred = trainer.predict(val_fold)
    oof_preds[val_idx] = np.argmax(val_pred.predictions, axis=1)

    test_pred = trainer.predict(tokenized_test)
    test_preds += np.argmax(test_pred.predictions, axis=1) / skf.n_splits

    fold_acc = accuracy_score(train.iloc[val_idx]['label'].values, oof_preds[val_idx])
    print(f"Fold {fold+1} Accuracy: {fold_acc:.5f}")

    shutil.rmtree(args.output_dir, ignore_errors=True)

    del model, trainer, train_fold, val_fold
    torch.cuda.empty_cache()
    gc.collect()

print(f"\nFinal OOF Accuracy: {accuracy_score(train['label'], oof_preds):.6f}")


=== Fold 1 ===


pytorch_model.bin:   0%|          | 0.00/669M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/669M [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss,Accuracy
1,0.663900,0.501355,0.782222
2,0.433800,0.443231,0.786667
3,0.393000,0.474493,0.788889
4,0.373800,0.461080,0.802222
5,0.298500,0.534294,0.793333
6,0.264100,0.500375,0.811111
7,0.237300,0.544947,0.824444
8,0.225000,0.538544,0.824444
9,0.167300,0.572758,0.806667
10,0.218900,0.567799,0.820000


Fold 1 Accuracy: 0.82444

=== Fold 2 ===


...

Fold 3 Accuracy: 0.80444

=== Fold 4 ===


Epoch,Training Loss,Validation Loss,Accuracy
1,0.651800,0.514026,0.768889
2,0.488400,0.415941,0.802222
3,0.416100,0.370757,0.828889
4,0.363800,0.359950,0.828889
5,0.316900,0.345588,0.851111
6,0.218500,0.385471,0.851111
7,0.256100,0.381732,0.842222
8,0.204600,0.411263,0.846667
9,0.182000,0.422171,0.846667
10,0.192600,0.428400,0.846667


Fold 4 Accuracy: 0.85111

=== Fold 5 ===


Epoch,Training Loss,Validation Loss,Accuracy
1,0.675000,0.496581,0.771111
2,0.501700,0.405250,0.820000
3,0.383900,0.369311,0.840000
4,0.365900,0.411118,0.840000
5,0.249500,0.527647,0.837778
6,0.238900,0.511711,0.835556
7,0.180200,0.476161,0.837778
8,0.201200,0.525902,0.842222
9,0.155400,0.530069,0.840000
10,0.154300,0.539023,0.842222


Fold 5 Accuracy: 0.84222

Final OOF Accuracy: 0.829333


In [8]:
sample=pd.read_csv('/kaggle/input/social-media-extremism-detection-challenge/sample_submission.csv')
final_test_preds = (test_preds >= 0.5).astype(int)

submission = sample.copy()
submission['Extremism_Label'] = [id2label[p] for p in final_test_preds]

submission.to_csv("submission.csv", index=False)
print("Submission saved!")
submission.head()

Submission saved!


,ID,Extremism_Label
0,2251,NON_EXTREMIST
1,2252,NON_EXTREMIST
2,2253,NON_EXTREMIST
3,2254,NON_EXTREMIST
4,2255,NON_EXTREMIST
